# Reproduce Classifier Results — LS8A (Formentera)

This notebook trains and evaluates the LightGBM benthic habitat classifier
on the processed per-site sample tables shipped in `data/`. It reproduces
the F1-Macro / Kappa / confusion-matrix results reported in the paper for
the LS8A (Formentera) Learning Site -- a 3-class drone-derived training
set (ANGIO / ROCK / SEDIMENT), with 130 spectral feature columns
(multi-temporal marine indices, t0-t3, plus their mean/std/range
aggregates).

**Not included**: the feature-extraction and drone-sample-extraction steps
that produced this table (raw Sentinel-2 -> ACOLITE correction -> spectral
feature extraction -> drone-raster resampling). Those are part of the
institute's internal pipeline. See the README for details.

The same notebook works unchanged on any other site's sample table -- just
point `SITE_FILE` at a different CSV.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path("..") / "src"))
from benthic_upscale import BenthicClassifier, BenthicMapper

DATA_DIR = Path("..") / "data"
RANDOM_SEED = 42

ModuleNotFoundError: No module named 'numpy'

## 1. Load a processed sample table

Each CSV contains one row per pixel sample, with spectral feature columns
plus a `class` label and a `region` identifier. To try a different site,
point `SITE_FILE` at another per-site table (e.g. `ls3c_samples.csv`).

In [ ]:
SITE_FILE = DATA_DIR / "ls8a_samples.csv"

samples = pd.read_csv(SITE_FILE)
non_feature_cols = {"class", "class_id", "region", "x_coord", "y_coord"}
feature_names = [c for c in samples.columns if c not in non_feature_cols]

print(f"Site        : {samples['region'].iloc[0]}")
print(f"Samples     : {len(samples):,}")
print(f"Features    : {len(feature_names)}")
print(f"Classes     : {sorted(samples['class'].unique())}")
samples.head()

## 2. Train / validation / test split

80% train+val / 20% held-out test, stratified by class.
85% / 15% split of train+val for early-stopping validation.

In [ ]:
X = samples[feature_names].values
y = samples["class"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15, random_state=RANDOM_SEED, stratify=y_tv
)

print(f"Train : {len(X_train):,}")
print(f"Val   : {len(X_val):,}")
print(f"Test  : {len(X_test):,}")

## 3. Train the classifier

In [ ]:
MODEL_PARAMS = {
    "n_estimators": 500,
    "max_depth": 15,
    "learning_rate": 0.05,
}

clf = BenthicClassifier(**MODEL_PARAMS)
clf.train(X_train, y_train, X_val, y_val, feature_names=feature_names)

## 4. Evaluate on the held-out test set

In [ ]:
results = clf.evaluate(X_test, y_test)

print("Test-set metrics")
print(f"  Accuracy      : {results['accuracy']:.4f}")
print(f"  F1-Macro      : {results['f1_macro']:.4f}")
print(f"  F1-Weighted   : {results['f1_weighted']:.4f}")
print(f"  Cohen's Kappa : {results['kappa']:.4f}")
print()
print(pd.DataFrame(results["classification_report"]).T)

In [ ]:
clf.plot_confusion_matrix(results, title=f"Confusion Matrix — {samples['region'].iloc[0]}")

In [ ]:
clf.plot_feature_importance(top_n=min(20, len(feature_names)))

## 5. (Optional) Full-stack prediction + habitat map

If you have a (H, W, F) feature stack for a full scene -- e.g. exported
from your own Sentinel-2 feature extraction, or provided as a `.npy` file
alongside the sample table -- you can reproduce the habitat map figure.
This step does not require raw imagery; it only needs an already-built
feature stack array.

In [ ]:
STACK_FILE = DATA_DIR / f"{samples['region'].iloc[0]}_feature_stack.npy"

if STACK_FILE.exists():
    feature_stack = np.load(STACK_FILE)
    mapper = BenthicMapper()

    pred_map = mapper.predict_full_image(feature_stack, clf, batch_size=50_000)
    mapper.plot_results(
        prediction_map=pred_map,
        classifier=clf,
        region=samples["region"].iloc[0],
    )
    areas = mapper.class_areas(pred_map, clf, pixel_size=10.0)
    print(areas.to_string(index=False))
else:
    print(f"No feature stack found at {STACK_FILE} -- skipping map reproduction.")
    print("(This is expected unless a stack file was provided for this site.)")